[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rudrite/kernels/blob/main/labs/stage-4/lab-4.2-ring-attention.ipynb)

# LAB·4.2 · Ring attention, composed

**Hardware:** the full algorithm runs anywhere (8 simulated devices); the remote-DMA kernel version needs a real slice.

The payoff of the whole track in one composition: LAB·3.1's monoid plus LAB·4.1's ring equals distributed attention, with nothing new invented. KV lives sharded across devices; each device streams every shard past its resident queries, rotating shards around the ring, and the carried (m, l, acc) state rescales exactly as it did on one chip. The algebra does not care where a block came from.

In [ ]:
import os
# simulate 8 devices when no real multi-chip slice is attached; must run before jax imports
if "COLAB_TPU_ADDR" not in os.environ:
    os.environ.setdefault("XLA_FLAGS", "--xla_force_host_platform_device_count=8")

import time
import numpy as np
import jax
import jax.numpy as jnp
from jax.sharding import Mesh, PartitionSpec as P
from jax.experimental.shard_map import shard_map
from functools import partial

devs = jax.devices()
print(jax.__version__, len(devs), "devices:", devs[0].platform)
ON_TPU = devs[0].platform == "tpu"

def check(name, got, want, tol=2e-2):
    err = float(jnp.abs(got.astype(jnp.float32) - want.astype(jnp.float32)).max())
    status = "ok" if err <= tol else "FAIL"
    print(f"{name}: max err {err:.3e} [{status}]")
    assert err <= tol, name


In [ ]:
# LAB 3.1's summarize/combine, verbatim: transport changes, algebra does not
def summarize(s_block, v_block):
    m = jnp.max(s_block, axis=-1)
    p = jnp.exp(s_block - m[..., None])
    return (m, jnp.sum(p, axis=-1), p @ v_block)

def combine(a, b):
    m1, l1, acc1 = a
    m2, l2, acc2 = b
    m = jnp.maximum(m1, m2)
    a1, a2 = jnp.exp(m1 - m), jnp.exp(m2 - m)
    return (m, l1 * a1 + l2 * a2, acc1 * a1[..., None] + acc2 * a2[..., None])

mesh = Mesh(np.array(devs), ("x",))
AXIS = len(devs)

@partial(shard_map, mesh=mesh,
         in_specs=(P("x", None), P("x", None), P("x", None)), out_specs=P("x", None))
def ring_attention(q, k, v):
    n = AXIS  # static mesh size from the closure
    perm = [(i, (i + 1) % n) for i in range(n)]
    state = summarize(q @ k.T, v)                    # my resident KV shard
    def hop(t, carry):
        state, k, v = carry
        k = jax.lax.ppermute(k, "x", perm)           # the shard rides the ring...
        v = jax.lax.ppermute(v, "x", perm)
        return combine(state, summarize(q @ k.T, v)), k, v   # ...while we fold it in
    state, _, _ = jax.lax.fori_loop(0, n - 1, hop, (state, k, v))
    m, l, acc = state
    return acc / l[..., None]

In [ ]:
SQ, SKV, D = 16 * AXIS, 32 * AXIS, 64
q = jax.random.normal(jax.random.key(0), (SQ, D))
k = jax.random.normal(jax.random.key(1), (SKV, D))
v = jax.random.normal(jax.random.key(2), (SKV, D))

got = ring_attention(q, k, v)
want = jax.nn.softmax(q @ k.T, axis=-1) @ v
check("ring attention == full attention", got, want, tol=1e-5)
print(f"distributed over {AXIS} devices, no device ever held more than 1/{AXIS} of KV")

## The kernel version, and what to notice

On a real slice, swap each `ppermute` hop for the LAB·4.1 remote DMA and put LAB·3.2's flash kernel inside the loop; the transfer of shard t+1 hides behind the attention math on shard t, which is the entire economics of the pattern. The site's EX·04 instrument animates it.

Then notice what you did not do: you never re-derived anything. Causal masking (LAB·3.4) composes in the same way, giving causal ring attention for free. When compositions keep coming out correct because one identity was proven once, you have learned the deepest lesson this track has: **the famous kernels are few theorems wearing many schedules.**

Gate criterion for the stage: LAB·4.1 bitwise + overlap; here, the composition check green plus, on a slice, a profile showing the hop hidden under compute.